[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/03_core_theory/core_theory_lab.ipynb)

# Core Theory Lab — Bias–Variance, Regularization, Optimizers

| | |
|---|---|
| **Companion to** | Session 3 — ML Fundamentals: Core Theory |
| **Runtime** | CPU only (synthetic data; no model downloads) |
| **Estimated time** | 30 minutes |
| **Last verified** | 2026-07-28 |

Session 3's claims are the kind an interviewer expects you to *state*. This notebook makes them things you have *seen*, which is what turns a memorized definition into a defended one.

You will produce, with your own runs:

1. The **bias–variance decomposition** measured empirically — not asserted — including the irreducible-noise floor.
2. **L1 vs L2 shrinkage paths**, showing why one zeroes coefficients and the other does not.
3. **Ridge = MAP under a Gaussian prior**, verified numerically to floating-point agreement.
4. **Optimizer trajectories** on a ravine, showing what momentum and adaptivity each fix.
5. **Adam vs AdamW**, showing that decoupling actually changes where you land.
6. The **ROC-vs-PR imbalance trap**, reproduced so you can quote the mechanism from memory.

> **Nothing here goes in the Metric Vault.** These are synthetic demonstrations of general theory, not measurements of your projects. Never present a number from this notebook as a result of your own work.

In [ ]:
%pip install -q "numpy>=1.26" "scikit-learn>=1.4" "matplotlib>=3.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(0)
NOISE_SD = 0.35          # the irreducible noise we inject; the floor we should recover
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("ready")

---
## Part 1 — Measuring the bias–variance decomposition

Lesson 1 states, for squared error:

$$\mathbb{E}\big[(y - \hat f(x))^2\big] = \underbrace{(f(x) - \mathbb{E}[\hat f(x)])^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\big[(\hat f(x) - \mathbb{E}[\hat f(x)])^2\big]}_{\text{Variance}} + \sigma^2$$

The expectations are over **training samples**. So to measure it, we resample the training set many times, fit a fresh model each time, and look at how the predictions scatter at a fixed test point.

The true function is a smooth curve; we fit polynomials of increasing degree.

In [ ]:
def true_f(x):
    """The ground-truth function. Smooth, non-polynomial in its own right."""
    return np.sin(1.6 * x) + 0.35 * x

def sample_train(n=25):
    x = RNG.uniform(-3, 3, size=n)
    y = true_f(x) + RNG.normal(0, NOISE_SD, size=n)
    return x, y

def fit_poly_predict(x_tr, y_tr, degree, x_te):
    """Least-squares polynomial fit -> predictions at x_te."""
    coeffs = np.polyfit(x_tr, y_tr, degree)
    return np.polyval(coeffs, x_te)

x_test = np.linspace(-2.8, 2.8, 120)
f_test = true_f(x_test)

# Show what a handful of fits look like at low / medium / high capacity.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, degree in zip(axes, [1, 4, 14]):
    for _ in range(12):
        x_tr, y_tr = sample_train()
        ax.plot(x_test, fit_poly_predict(x_tr, y_tr, degree, x_test),
                color="tab:blue", alpha=0.30, linewidth=1)
    ax.plot(x_test, f_test, color="black", linewidth=2.5, label="true f(x)")
    ax.set_title(f"degree {degree}")
    ax.set_ylim(-3.5, 3.5)
    ax.legend(loc="upper left", fontsize=8)
fig.suptitle("12 fits, each on a different training sample", y=1.02)
plt.tight_layout(); plt.show()

**Read the picture before the numbers.** Degree 1 lines sit tightly bunched but in the wrong shape — that bunching *is* low variance, the wrongness *is* high bias. Degree 14 curves pass near the data but fan out wildly between samples — low bias, high variance. Degree 4 is visibly the compromise.

Now measure it. For each capacity we run many independent training samples, then compute the three terms directly from their definitions.

In [ ]:
def bias_variance(degree, n_sims=300):
    """Empirical bias^2 / variance / total error at each test point, averaged over x."""
    preds = np.empty((n_sims, x_test.size))
    for s in range(n_sims):
        x_tr, y_tr = sample_train()
        preds[s] = fit_poly_predict(x_tr, y_tr, degree, x_test)

    mean_pred = preds.mean(axis=0)
    bias_sq = np.mean((mean_pred - f_test) ** 2)
    variance = np.mean(preds.var(axis=0))
    total = bias_sq + variance + NOISE_SD ** 2
    return bias_sq, variance, total

degrees = np.arange(1, 16)
rows = np.array([bias_variance(d) for d in degrees])
bias_sq, variance, total = rows[:, 0], rows[:, 1], rows[:, 2]

best = degrees[np.argmin(total)]
print(f"irreducible noise floor  sigma^2 = {NOISE_SD**2:.4f}")
print(f"minimum expected error at degree {best}: {total.min():.4f}")
print()
print(f"{'deg':>4} {'bias^2':>9} {'variance':>10} {'total':>9}")
for d, b, v, t in zip(degrees, bias_sq, variance, total):
    mark = "  <-- best" if d == best else ""
    print(f"{d:>4} {b:>9.4f} {v:>10.4f} {t:>9.4f}{mark}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(degrees, bias_sq, "o-", label="Bias$^2$", color="tab:orange")
ax.plot(degrees, variance, "s-", label="Variance", color="tab:blue")
ax.plot(degrees, total, "^-", label="Total expected error", color="tab:red", linewidth=2)
ax.axhline(NOISE_SD ** 2, linestyle="--", color="gray",
           label=f"Irreducible $\\sigma^2$ = {NOISE_SD**2:.3f}")
ax.axvline(best, linestyle=":", color="black", alpha=0.6)
ax.set_xlabel("polynomial degree (capacity)"); ax.set_ylabel("error")
ax.set_yscale("log"); ax.set_title("The bias-variance tradeoff, measured")
ax.legend(); plt.tight_layout(); plt.show()

**What you just produced.** Bias² falls monotonically with capacity, variance rises, and their sum is U-shaped with a minimum at an intermediate degree. The dashed line is the noise floor: the red curve never goes below it, no matter the capacity. That floor is the σ² term — the part of the target no model can predict.

**The interview payoff.** When asked "explain the bias–variance tradeoff," you can now say *"I've measured this — bias² falls, variance rises, the sum is U-shaped, and there's a noise floor the total never crosses"*, which lands very differently from reciting the formula.

> ⚠️ **Where this picture stops being the whole story.** The classic U-curve describes the *under-parameterized* regime. Past the interpolation threshold, very over-parameterized models show **double descent** — test error falls again (Belkin et al. 2019). Polynomial least-squares at n=25 can't show that; mentioning the limitation unprompted is the Gold-tier move.

---
## Part 2 — L1 vs L2: watching shrinkage happen

Lesson 1 claims L1 drives coefficients to *exactly* zero while L2 only shrinks them toward zero. That difference is geometric, and you can watch it directly by sweeping the penalty strength and plotting every coefficient's path.

We build a deliberately sparse problem: 12 features, only 3 of which actually matter.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

n_samples, n_features = 60, 12
X = RNG.normal(size=(n_samples, n_features))
true_w = np.zeros(n_features)
true_w[[1, 4, 7]] = [2.5, -1.8, 1.2]          # only 3 real signals
y = X @ true_w + RNG.normal(0, 0.5, size=n_samples)

alphas = np.logspace(-3, 1.4, 60)
ridge_paths = np.array([Ridge(alpha=a).fit(X, y).coef_ for a in alphas])
lasso_paths = np.array([Lasso(alpha=a, max_iter=20000).fit(X, y).coef_ for a in alphas])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, paths, name in [(axes[0], ridge_paths, "L2 (Ridge)"),
                        (axes[1], lasso_paths, "L1 (Lasso)")]:
    for j in range(n_features):
        signal = true_w[j] != 0
        ax.plot(alphas, paths[:, j],
                color="tab:red" if signal else "tab:gray",
                linewidth=2 if signal else 1,
                alpha=1.0 if signal else 0.55)
    ax.set_xscale("log"); ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("penalty strength $\\alpha$"); ax.set_title(name)
axes[0].set_ylabel("coefficient value")
fig.suptitle("Red = truly non-zero features, grey = noise features", y=1.02)
plt.tight_layout(); plt.show()

# Count exact zeros at a moderate penalty.
mid = len(alphas) // 2
print(f"at alpha = {alphas[mid]:.3f}")
print(f"  Ridge: {np.sum(np.abs(ridge_paths[mid]) < 1e-8):>2} coefficients exactly zero")
print(f"  Lasso: {np.sum(np.abs(lasso_paths[mid]) < 1e-8):>2} coefficients exactly zero")

**The mechanism you can now describe.** Ridge paths approach zero asymptotically but never arrive — every grey line stays slightly alive. Lasso paths hit zero *exactly* and stay there, switching features off one at a time as the penalty grows.

The geometric reason, worth one sentence in an interview: the L1 constraint region is a diamond with corners **on the axes**, so the loss contour typically first touches it at a corner — where some coordinates are exactly zero. The L2 region is a sphere with no corners, so the touch point almost never has an exact zero.

That is why **L1 = feature selection** and **L2 = stable shrinkage**.

---
## Part 3 — Verifying that Ridge *is* MAP under a Gaussian prior

Lesson 1's derivation claims:

$$\hat w_{\text{MAP}} = \arg\min_w \; \lVert y - Xw\rVert^2 + \lambda\lVert w\rVert^2 \quad\text{with}\quad \lambda = \frac{\sigma^2}{\tau^2}$$

for a Gaussian likelihood with noise σ² and a Gaussian prior $w \sim \mathcal{N}(0, \tau^2 I)$.

This is a claim you can *check*, not just assert. Below we compute the MAP estimate two ways — the closed-form Bayesian posterior mean, and sklearn's Ridge with the corresponding α — and compare them.

In [ ]:
sigma2, tau2 = 0.5 ** 2, 1.0 ** 2      # noise variance, prior variance
lam = sigma2 / tau2                     # the correspondence the derivation predicts

# 1. Bayesian posterior mean (MAP for a Gaussian posterior), closed form:
#    w_MAP = (X^T X + (sigma^2/tau^2) I)^-1 X^T y
A = X.T @ X + lam * np.eye(n_features)
w_map = np.linalg.solve(A, X.T @ y)

# 2. Ridge regression with alpha = lambda (no intercept, to match exactly)
w_ridge = Ridge(alpha=lam, fit_intercept=False).fit(X, y).coef_

max_abs_diff = np.max(np.abs(w_map - w_ridge))
print(f"lambda = sigma^2 / tau^2 = {lam:.4f}")
print(f"max |w_MAP - w_ridge|    = {max_abs_diff:.2e}")
print("identical to floating point:", np.allclose(w_map, w_ridge))
print()
print(f"{'j':>3} {'true':>8} {'MAP':>9} {'Ridge':>9}")
for j in range(n_features):
    print(f"{j:>3} {true_w[j]:>8.3f} {w_map[j]:>9.3f} {w_ridge[j]:>9.3f}")

**They agree to machine precision** — the two are the same estimator wearing different vocabulary. This is the single most useful connection in Session 3, because it lets you answer "why does regularization work?" at a level most candidates don't reach: *a penalty is a prior, and the penalty strength is the ratio of noise variance to prior variance.*

Note what that ratio means physically: **more noise (σ² up) → stronger regularization**, and **a tighter prior belief (τ² down) → stronger regularization**. Both match intuition, which is a good sign you've understood rather than memorized.

---
## Part 4 — Optimizer trajectories: what momentum and adaptivity each fix

Lesson 2 walks SGD → momentum → adaptive → Adam → AdamW. Each step fixes a specific pathology, and the classic way to *see* it is an ill-conditioned quadratic — a "ravine" that is steep in one direction and nearly flat in another.

$$L(w) = \tfrac{1}{2}\left(a\,w_1^2 + b\,w_2^2\right), \qquad a = 20 \gg b = 1$$

Gradient descent takes a step of $\eta a w_1$ in the steep direction and $\eta b w_2$ in the flat one. That is the bind: the steep direction caps how large $\eta$ can be (past $\eta > 2/a$ it diverges), and at any $\eta$ small enough to be safe there, the flat direction barely moves. **One learning rate, two curvatures, no good compromise** — this is exactly the geometry every optimizer after SGD is trying to fix.

Watch each optimizer's path.

In [ ]:
A_CURV, B_CURV = 20.0, 1.0     # curvature: steep in w1, flat in w2

def loss(w):  return 0.5 * (A_CURV * w[0] ** 2 + B_CURV * w[1] ** 2)
def grad(w):  return np.array([A_CURV * w[0], B_CURV * w[1]])

def run(optimizer, lr, steps=60, w0=(-2.6, 2.2), **kw):
    """Return the trajectory of `optimizer` from w0."""
    w = np.array(w0, dtype=float)
    m = np.zeros(2); v = np.zeros(2)
    path = [w.copy()]
    for t in range(1, steps + 1):
        g = grad(w)
        if optimizer == "sgd":
            w -= lr * g
        elif optimizer == "momentum":
            m = kw.get("beta", 0.9) * m + g
            w -= lr * m
        elif optimizer == "rmsprop":
            v = 0.99 * v + 0.01 * g ** 2
            w -= lr * g / (np.sqrt(v) + 1e-8)
        elif optimizer in ("adam", "adamw"):
            b1, b2 = 0.9, 0.999
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g ** 2
            mhat, vhat = m / (1 - b1 ** t), v / (1 - b2 ** t)
            step = mhat / (np.sqrt(vhat) + 1e-8)
            if optimizer == "adamw":                 # decoupled decay
                w -= lr * (step + kw.get("wd", 0.0) * w)
            else:
                w -= lr * step
        path.append(w.copy())
    return np.array(path)

# SGD and momentum share a learning rate -- that is the fair comparison, since
# momentum's whole claim is "more progress per step at the same lr".
# The adaptive methods normalise the gradient, so they take a larger lr.
runs = {
    "SGD (lr=0.02)":            run("sgd", 0.02),
    "SGD+momentum (lr=0.02)":   run("momentum", 0.02),
    "RMSProp (lr=0.10)":        run("rmsprop", 0.10),
    "Adam (lr=0.10)":           run("adam", 0.10),
}

grid = np.linspace(-3, 3, 200)
G1, G2 = np.meshgrid(grid, grid)
Z = 0.5 * (A_CURV * G1 ** 2 + B_CURV * G2 ** 2)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(G1, G2, Z, levels=np.logspace(-1, 2.2, 18), colors="lightgray", linewidths=0.8)
for (name, path), color in zip(runs.items(), ["tab:red", "tab:orange", "tab:green", "tab:blue"]):
    ax.plot(path[:, 0], path[:, 1], "-o", markersize=2.5, linewidth=1.5, label=name, color=color)
ax.plot(0, 0, "k*", markersize=15, label="optimum")
ax.set_xlabel("$w_1$ (steep direction)"); ax.set_ylabel("$w_2$ (flat direction)")
ax.set_title("60 steps on an ill-conditioned quadratic")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f"{'optimizer':<26} {'L@10':>10} {'L@20':>10} {'L@60':>10}")
for name, path in runs.items():
    L = [loss(w) for w in path]
    print(f"{name:<26} {L[10]:>10.2e} {L[20]:>10.2e} {L[60]:>10.2e}")

**Read the columns, not just the picture.** At a *matched* learning rate, momentum ends roughly 6× lower than plain SGD — that is its claim, demonstrated. The adaptive methods are in a different league on this problem.

- **SGD** creeps along the flat $w_2$ direction: its step there is $\eta b$ with $b=1$, so it barely moves, while $\eta a$ with $a=20$ is what limits how large $\eta$ can be. One learning rate cannot serve two curvatures — that is the ravine pathology.
- **+ Momentum** accumulates velocity along the consistent flat direction, so the same $\eta$ travels much further. **Look at the L@10 column:** momentum is *worse* early — it builds speed and overshoots — then converges well past SGD. That overshoot-then-settle signature is characteristic and worth recognising.
- **RMSProp** attacks it from the other side: dividing by $\sqrt{\text{avg }g^2}$ gives each coordinate its *own* effective step, so the flat direction stops being starved. On a clean quadratic this is dramatic.
- **Adam** combines both, beating SGD by step 60 — but notice it **stops improving** rather than converging to machine zero.

> ⚠️ **Why Adam plateaus here, and what it teaches.** Adam's update is $\eta \cdot \hat m/\sqrt{\hat v}$ — a *normalised* step whose size stays near $\eta$ even as the gradient shrinks. Near the optimum it therefore hovers at scale $\sim\eta$ instead of settling. This is not a bug in the demo; it is exactly why production training **decays the learning rate**, and it is the concrete evidence behind Lesson 2's claim that *the schedule often matters more than the optimizer*.

Compressed for an interview: *"momentum fixes oscillation and slow flat directions, adaptivity fixes per-parameter scaling, Adam does both — and you still need an LR schedule, because a normalised step doesn't shrink on its own."*

### Adam vs AdamW — the decoupling is not cosmetic

Lesson 2's Drill 3: in Adam, an L2 term added to the loss gets divided by $\sqrt{\hat v}$ along with everything else, so the decay is *normalised away*. AdamW applies decay directly to the weights, outside the adaptive scaling.

The sharpest test is not "are the trajectories different" but **"does turning the knob do anything?"** Sweep the decay coefficient $\lambda$ over two orders of magnitude under both implementations and look at the final weight.

In [ ]:
def train_1d(mode, wd, steps=200, lr=0.01, grad_scale=2.0, w0=1.0):
    """Minimise 0.5*grad_scale*w^2 under two different decay implementations."""
    w = w0
    m = v = 0.0
    b1, b2, eps = 0.9, 0.999, 1e-8
    for t in range(1, steps + 1):
        # Adam-with-L2 folds the decay INTO the gradient (so it gets normalised);
        # AdamW keeps the gradient clean and decays the weight separately.
        g = grad_scale * w + (wd * w if mode == "adam_l2" else 0.0)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        step = (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)
        w = w - lr * (step + (wd * w if mode == "adamw" else 0.0))
    return abs(w)

wds = [0.0, 0.01, 0.1, 0.5, 1.0]
adam_l2 = [train_1d("adam_l2", wd) for wd in wds]
adamw   = [train_1d("adamw",   wd) for wd in wds]

print("Does turning up weight decay actually shrink the weight?")
print(f"{'weight decay':>13} {'Adam + L2-in-loss':>20} {'AdamW (decoupled)':>20}")
for wd, a, w_ in zip(wds, adam_l2, adamw):
    print(f"{wd:>13} {a:>20.5f} {w_:>20.5f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(wds, adam_l2, "o-", linewidth=2, color="tab:orange", label="Adam + L2-in-loss")
ax.plot(wds, adamw,  "s-", linewidth=2, color="tab:blue",   label="AdamW (decoupled)")
ax.set_xlabel("weight-decay coefficient $\\lambda$")
ax.set_ylabel("final $|w|$"); ax.set_yscale("log")
ax.set_title("Only decoupled decay actually responds to $\\lambda$")
ax.legend(); plt.tight_layout(); plt.show()

**Look at the left column: it does not move.** From $\lambda = 0$ to $\lambda = 1.0$ — a hundred-fold change — Adam-with-L2 lands on the *same* final weight to five decimal places. The decay term entered the gradient, then got divided by $\sqrt{\hat v}$, which is itself proportional to that same gradient — so the knob normalises itself away. Meanwhile AdamW's weight shrinks monotonically as you turn $\lambda$ up, which is what the hyperparameter is supposed to do.

This reproduces the practitioner complaint that motivated the AdamW paper: *"weight decay doesn't seem to do anything when I use Adam."* It wasn't doing anything.

**The one sentence to carry into the interview:** "L2 regularization and weight decay are equivalent for plain SGD, but **not** for adaptive optimizers — in Adam the L2 term gets normalised by the second-moment estimate, so AdamW decouples it and applies $\eta\lambda w$ directly to the weights." That is what Drill 3 is really testing, and you have now watched it happen.

---
## Part 6 — The ROC-vs-PR imbalance trap

Lesson 3 claims a model can post an excellent ROC-AUC and still be practically useless when positives are rare, because FPR's denominator is the whole (huge) negative class. Reproduce it once and you will never mis-answer this question.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

def imbalanced_scores(n=20000, prevalence=0.005, separation=2.2):
    """Scores from a decent-but-not-perfect ranker on a rare-positive problem."""
    n_pos = int(n * prevalence)
    y = np.zeros(n, dtype=int); y[:n_pos] = 1
    s = np.where(y == 1, RNG.normal(separation, 1.0, n), RNG.normal(0.0, 1.0, n))
    return y, s

y_true, scores = imbalanced_scores()
prevalence = y_true.mean()
roc_auc = roc_auc_score(y_true, scores)
pr_auc  = average_precision_score(y_true, scores)

print(f"positives: {y_true.sum()} / {y_true.size}  (prevalence {prevalence:.3%})")
print(f"ROC-AUC          = {roc_auc:.3f}   <- looks excellent")
print(f"PR-AUC (avg prec)= {pr_auc:.3f}   <- the honest picture")
print(f"random baseline: ROC-AUC = 0.500, PR-AUC = prevalence = {prevalence:.3f}")

# What a user actually experiences at a high-recall operating point.
thr = np.quantile(scores, 1 - 0.02)          # flag the top 2% of cases
flagged = scores >= thr
tp = int((flagged & (y_true == 1)).sum()); fp = int((flagged & (y_true == 0)).sum())
print(f"\nOperating point: flag top 2% ({flagged.sum()} alerts)")
print(f"  true positives : {tp}")
print(f"  false positives: {fp}")
print(f"  precision      : {tp / max(tp + fp, 1):.1%}  <- ~{fp / max(tp,1):.0f} false alarms per real hit")
print(f"  recall         : {tp / y_true.sum():.1%}")

In [ ]:
fpr, tpr, _ = roc_curve(y_true, scores)
prec, rec, _ = precision_recall_curve(y_true, scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(fpr, tpr, color="tab:blue", linewidth=2)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title(f"ROC curve — AUC {roc_auc:.3f} (flattering)")

axes[1].plot(rec, prec, color="tab:red", linewidth=2)
axes[1].axhline(prevalence, linestyle="--", color="gray",
                label=f"random baseline = prevalence = {prevalence:.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR curve — AP {pr_auc:.3f} (honest)")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

**The mechanism, now visible.** The ROC curve hugs the top-left because even thousands of false positives are a tiny *fraction* of ~20,000 negatives. The PR curve collapses because those same false positives are the overwhelming majority of what you actually flagged.

One sentence for the interview: **"FPR divides by all negatives — a huge denominator — while precision divides by predicted positives, so under heavy imbalance ROC stays flattering while precision reflects what the user experiences."**

Always quote PR-AUC against its baseline (the prevalence), never alone.

---
## What you can now claim (and what you cannot)

**Can claim** — you have *seen* these, so you may say "I've worked through this":

- Bias² falls and variance rises with capacity; their sum is U-shaped above an irreducible noise floor.
- L1 zeroes coefficients exactly; L2 shrinks asymptotically. The corner-vs-sphere geometry is why.
- Ridge and Gaussian-prior MAP are the same estimator, with $\lambda = \sigma^2/\tau^2$.
- Momentum fixes oscillation; adaptivity fixes per-parameter scaling; Adam does both; AdamW's decoupled decay lands somewhere genuinely different.
- ROC-AUC can look excellent under heavy imbalance while precision collapses.

**Cannot claim** — everything above is synthetic. These are demonstrations of general theory, **not** measurements of your projects. Your ESCI macro-F1 and calibration ECE come from your own run logs and nowhere else.

### Next

Return to [Lesson 4 — Rapid-Fire Rehearsal & Mock Breadth Round](04_rapid_fire_and_mock_round.md) and run the mock round. The numbers you just watched appear will make the caveats far easier to say under time pressure.